# BBC News classification with custom BoW and TF-IDF

Mariam Ghazaryan

This assignment implements preprocessing, vocabulary construction, Bag of Words, term frequency and inverse document frequency from scratch. It compares the resulting representations in a five-category news classifier and analyzes category keywords and common versus rare terms.

Place `bbc-text.csv` beside this notebook. Install dependencies with `pip install -r requirements.txt`. The first run downloads NLTK English stopwords if missing.

## Method and formulas

1. Convert text to lowercase, replace punctuation and digits with token boundaries, tokenize alphabetic words and remove NLTK English stopwords. No stemming or lemmatization is required here.
2. Remove duplicate **preprocessed** documents before a stratified 80/20 train/test split with seed 42. This prevents identical inputs appearing in both sets. Near-duplicate articles may still exist.
3. Build a sorted vocabulary from training documents only. Test words outside it are ignored.
4. BoW(t,d) = count(t,d). TF(t,d) = count(t,d), following the reference PDF's core examples. DF(t) counts training documents containing t, once per document. IDF(t) = ln(N / DF(t)). TF-IDF(t,d) = TF(t,d) × IDF(t).
5. L2-normalize TF-IDF vectors for classification, leaving empty vectors zero. Category analysis uses **unnormalized** count × IDF weights; its averages include zeros for category documents without the word. These averages can favor words repeated in long articles.
6. Use the same Multinomial Naive Bayes classifier (alpha=1) for both representations and report accuracy, macro F1 and per-class metrics on the held-out test set. The classifier, split and metrics use scikit-learn; vectorization is custom. This is a fixed baseline with no tuning against test labels.

The reference discusses multiple TF/IDF variants. This solution explicitly chooses raw TF and unsmoothed IDF, rather than a library default. Training vocabulary terms have DF ≥ 1, so no smoothing is needed.

## Custom implementation

In [1]:
"""Custom BoW and TF-IDF for BBC News classification.

Only the split, classifier and evaluation use scikit-learn. No library text
vectorizer or TF-IDF transformer is used. Run from this script's directory.
"""
from collections import Counter
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

try:
    STOP_WORDS = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True, raise_on_error=True)
    STOP_WORDS = set(stopwords.words('english'))


def preprocess(text):
    """Lowercase, replace punctuation/digits with spaces, tokenize, filter."""
    return [word for word in re.findall(r'[a-z]+', str(text).lower())
            if word not in STOP_WORDS]


def build_vocabulary(documents):
    """Assign deterministic column indices to training words only."""
    return {word: i for i, word in enumerate(sorted(
        {word for document in documents for word in document}))}


def document_to_bow(tokens, vocabulary):
    """One dense count vector; unseen words are ignored."""
    vector = np.zeros(len(vocabulary), dtype=np.int32)
    for word, count in Counter(tokens).items():
        if word in vocabulary:
            vector[vocabulary[word]] = count
    return vector


def bow_matrix(documents, vocabulary):
    """Construct a sparse matrix directly from custom counts."""
    values, columns, offsets = [], [], [0]
    for tokens in documents:
        for word, count in sorted(Counter(tokens).items()):
            if word in vocabulary:
                columns.append(vocabulary[word])
                values.append(count)
        offsets.append(len(values))
    return csr_matrix((np.array(values, dtype=np.float64), columns, offsets),
                      shape=(len(documents), len(vocabulary)))


def calculate_tf(tokens):
    """TF is raw count, matching the core worked examples in the PDF."""
    return dict(Counter(tokens))


def calculate_idf(training_documents, vocabulary):
    """IDF(t) = ln(N / DF(t)); a document counts each term once."""
    document_frequency = np.zeros(len(vocabulary), dtype=np.int32)
    for tokens in training_documents:
        for word in set(tokens):
            if word in vocabulary:
                document_frequency[vocabulary[word]] += 1
    if len(training_documents) == 0 or np.any(document_frequency == 0):
        raise ValueError('Vocabulary must come from a nonempty training corpus.')
    return np.log(len(training_documents) / document_frequency), document_frequency


def document_to_tfidf(tokens, vocabulary, idf):
    """Unnormalized TF-IDF vector: raw term count times training IDF."""
    vector = np.zeros(len(vocabulary), dtype=float)
    for word, frequency in calculate_tf(tokens).items():
        if word in vocabulary:
            index = vocabulary[word]
            vector[index] = frequency * idf[index]
    return vector


def tfidf_matrix(counts, idf, normalize=False):
    """Multiply nonzero custom counts by IDF; optionally L2-normalize."""
    result = counts.copy().astype(float)
    result.data *= idf[result.indices]
    result.eliminate_zeros()
    if normalize:
        norms = np.sqrt(np.asarray(result.multiply(result).sum(axis=1)).ravel())
        for row in range(result.shape[0]):
            if norms[row] > 0:
                start, end = result.indptr[row:row + 2]
                result.data[start:end] /= norms[row]
    return result


def verify_vectorizers():
    """Hand-calculated fixture checks counts, DF, IDF, OOV and empty text."""
    docs = [['cat', 'cat', 'dog'], ['dog', 'bird'], []]
    vocabulary = build_vocabulary(docs)
    assert list(vocabulary) == ['bird', 'cat', 'dog']
    assert calculate_tf(docs[0]) == {'cat': 2, 'dog': 1}
    assert document_to_bow(docs[0], vocabulary).tolist() == [0, 2, 1]
    idf, df = calculate_idf(docs, vocabulary)
    np.testing.assert_array_equal(df, [1, 1, 2])
    np.testing.assert_allclose(idf, [np.log(3), np.log(3), np.log(1.5)])
    counts = bow_matrix(docs, vocabulary)
    np.testing.assert_array_equal(counts.toarray(), [[0, 2, 1], [1, 0, 1], [0, 0, 0]])
    vectors = tfidf_matrix(counts, idf)
    np.testing.assert_allclose(vectors[0].toarray().ravel(), [0, 2*np.log(3), np.log(1.5)])
    np.testing.assert_allclose(document_to_tfidf(docs[0], vocabulary, idf), vectors[0].toarray().ravel())
    assert not document_to_bow(['unseen'], vocabulary).any()
    normalized = tfidf_matrix(counts, idf, normalize=True)
    np.testing.assert_allclose(np.asarray(normalized.multiply(normalized).sum(axis=1)).ravel(), [1, 1, 0])
    assert preprocess('The CAT, and dog! 123') == ['cat', 'dog']
    print('All hand-calculated vectorizer checks passed.')


def run_experiment(data_path='bbc-text.csv', output_dir='results'):
    verify_vectorizers()
    output = Path(output_dir)
    output.mkdir(exist_ok=True, parents=True)
    raw = pd.read_csv(data_path)
    if raw[['text', 'category']].isna().any().any():
        raise ValueError('Missing texts or labels must be handled before training.')
    data = raw.copy()
    data['source_row'] = np.arange(len(data))
    # Deduplicate the actual preprocessed input to prevent exact overlap.
    data['tokens'] = data.text.map(preprocess)
    data['clean_text'] = data.tokens.map(' '.join)
    if data.groupby('clean_text').category.nunique().max() > 1:
        raise ValueError('Identical cleaned documents have conflicting labels.')
    data = data.drop_duplicates('clean_text').reset_index(drop=True)
    train, test = train_test_split(data, test_size=0.2, random_state=42,
                                   stratify=data.category)
    train, test = train.reset_index(drop=True), test.reset_index(drop=True)
    assert set(train.clean_text).isdisjoint(test.clean_text)
    vocabulary = build_vocabulary(train.tokens)
    words = np.array(list(vocabulary))
    idf, df = calculate_idf(train.tokens.tolist(), vocabulary)
    train_bow = bow_matrix(train.tokens, vocabulary)
    test_bow = bow_matrix(test.tokens, vocabulary)
    # Unnormalized weights are used for interpretable category averages.
    raw_tfidf = tfidf_matrix(train_bow, idf)
    train_tfidf = tfidf_matrix(train_bow, idf, normalize=True)
    test_tfidf = tfidf_matrix(test_bow, idf, normalize=True)
    scores, reports, predictions = [], {}, test[['source_row', 'category']].copy()
    labels = sorted(data.category.unique())
    for name, x_train, x_test in [('BoW', train_bow, test_bow),
                                  ('TF-IDF', train_tfidf, test_tfidf)]:
        model = MultinomialNB(alpha=1.0).fit(x_train, train.category)
        predicted = model.predict(x_test)
        scores.append({'representation': name, 'accuracy': accuracy_score(test.category, predicted),
                       'macro_f1': f1_score(test.category, predicted, average='macro')})
        reports[name] = classification_report(test.category, predicted, output_dict=True, zero_division=0)
        predictions[name] = predicted
        pd.DataFrame(confusion_matrix(test.category, predicted, labels=labels),
                     index=labels, columns=labels).to_csv(output/f'{name.lower().replace("-", "_")}_confusion_matrix.csv')
    top_rows = []
    for category in labels:
        averages = np.asarray(raw_tfidf[train.category.eq(category).to_numpy()].mean(axis=0)).ravel()
        for rank, index in enumerate(np.argsort(-averages, kind='stable')[:10], 1):
            top_rows.append({'category': category, 'rank': rank, 'word': words[index],
                             'average_tfidf': averages[index]})
    top_words = pd.DataFrame(top_rows)
    # Mean raw TF over ALL training documents, including absent terms as zero.
    stats = pd.DataFrame({'word': words, 'mean_tf': np.asarray(train_bow.mean(axis=0)).ravel(),
                          'document_frequency': df, 'idf': idf})
    # State explicit relative cutoffs rather than calling arbitrary words high/low.
    low_idf = stats.idf.quantile(0.10)
    high_idf = stats.idf.quantile(0.90)
    common = stats[stats.idf <= low_idf].sort_values(['mean_tf', 'word'], ascending=[False, True]).head(10)
    rare = stats[stats.idf >= high_idf].sort_values(['mean_tf', 'word']).head(10)
    results = pd.DataFrame(scores)
    summary = {'source_rows': len(raw), 'exact_raw_duplicates': int(raw.text.duplicated().sum()),
               'duplicates_removed_after_preprocessing': len(raw)-len(data), 'unique_documents': len(data),
               'train_documents': len(train), 'test_documents': len(test), 'vocabulary_size': len(vocabulary),
               'seed': 42, 'idf_low_cutoff': low_idf, 'idf_high_cutoff': high_idf,
               'test_oov_token_count': sum(w not in vocabulary for doc in test.tokens for w in doc),
               'test_token_count': sum(map(len, test.tokens))}
    results.to_csv(output/'classification_metrics.csv', index=False)
    top_words.to_csv(output/'top_category_words.csv', index=False)
    stats.to_csv(output/'training_word_statistics.csv', index=False)
    common.to_csv(output/'high_tf_low_idf.csv', index=False)
    rare.to_csv(output/'low_tf_high_idf.csv', index=False)
    predictions.to_csv(output/'test_predictions.csv', index=False)
    data[['source_row','category','clean_text']].to_csv(output/'preprocessed_articles.csv',index=False)
    pd.concat([train[['source_row','category']].assign(split='train'),
               test[['source_row','category']].assign(split='test')]).to_csv(output/'split_manifest.csv',index=False)
    (output/'summary.json').write_text(json.dumps(summary, indent=2))
    (output/'classification_reports.json').write_text(json.dumps(reports, indent=2))
    print(json.dumps(summary, indent=2))
    print('\nClassification results\n', results.to_string(index=False))
    return summary, results, top_words, common, rare, reports




## Run and validate
A hand-calculated three-document example checks counts, IDF, repeated words, unknown words and zero vectors before processing the BBC data.

In [2]:
summary, metrics, top_words, common, rare, reports = run_experiment()

All hand-calculated vectorizer checks passed.


{
  "source_rows": 2225,
  "exact_raw_duplicates": 99,
  "duplicates_removed_after_preprocessing": 106,
  "unique_documents": 2119,
  "train_documents": 1695,
  "test_documents": 424,
  "vocabulary_size": 25566,
  "seed": 42,
  "idf_low_cutoff": 4.43970574626056,
  "idf_high_cutoff": 7.43543801981455,
  "test_oov_token_count": 3151,
  "test_token_count": 90577
}

Classification results
 representation  accuracy  macro_f1
           BoW  0.969340  0.968598
        TF-IDF  0.964623  0.964361


## Dataset and evaluation

In [3]:
data = pd.read_csv('bbc-text.csv')
print('Original class distribution:')
print(data.category.value_counts().to_string())
print('\nAccuracy and macro F1:')
display(metrics)

Original class distribution:
category
sport            511
business         510
politics         417
tech             401
entertainment    386

Accuracy and macro F1:


,representation,accuracy,macro_f1
0,BoW,0.969340,0.968598
1,TF-IDF,0.964623,0.964361


In [4]:
for representation, report in reports.items():
    print(representation, 'classification report')
    display(pd.DataFrame(report).T)
    print('Confusion matrix: rows = actual, columns = predicted')
    display(pd.read_csv('results/' + representation.lower().replace('-', '_') + '_confusion_matrix.csv', index_col=0))

BoW classification report


,precision,recall,f1-score,support
business,0.949495,0.940000,0.944724,100.00000
entertainment,0.986111,0.959459,0.972603,74.00000
politics,0.975309,0.975309,0.975309,81.00000
sport,1.000000,1.000000,1.000000,100.00000
tech,0.930556,0.971014,0.950355,69.00000
accuracy,0.969340,0.969340,0.969340,0.96934
macro avg,0.968294,0.969157,0.968598,424.00000
weighted avg,0.969646,0.969340,0.969385,424.00000


Confusion matrix: rows = actual, columns = predicted


,business,entertainment,politics,sport,tech
business,94,1,2,0,3
entertainment,1,71,0,0,2
politics,2,0,79,0,0
sport,0,0,0,100,0
tech,2,0,0,0,67


TF-IDF classification report


,precision,recall,f1-score,support
business,0.906542,0.970000,0.937198,100.000000
entertainment,1.000000,0.932432,0.965035,74.000000
politics,0.963415,0.975309,0.969325,81.000000
sport,0.990099,1.000000,0.995025,100.000000
tech,0.984615,0.927536,0.955224,69.000000
accuracy,0.964623,0.964623,0.964623,0.964623
macro avg,0.968934,0.961055,0.964361,424.000000
weighted avg,0.966130,0.964623,0.964766,424.000000


Confusion matrix: rows = actual, columns = predicted


,business,entertainment,politics,sport,tech
business,97,0,2,0,1
entertainment,4,69,1,0,0
politics,2,0,79,0,0
sport,0,0,0,100,0
tech,4,0,0,1,64


## Top 10 words by average category TF-IDF
All statistics below use only training data. Results are shown for every category, covering the requested analysis for a given category.

In [5]:
for category in sorted(top_words.category.unique()):
    print('Category:', category)
    display(top_words[top_words.category.eq(category)][['rank','word','average_tfidf']].reset_index(drop=True))

Category: business


,rank,word,average_tfidf
0,1,bn,2.888380
1,2,bank,1.886782
2,3,oil,1.795014
3,4,economy,1.657285
4,5,yukos,1.627479
5,6,growth,1.542761
6,7,sales,1.539326
7,8,china,1.503371
8,9,market,1.480926
9,10,us,1.475860


Category: entertainment


,rank,word,average_tfidf
0,1,film,4.671749
1,2,best,2.882565
2,3,music,2.814733
3,4,awards,2.062738
4,5,band,1.861622
5,6,award,1.797246
6,7,song,1.750192
7,8,show,1.698880
8,9,album,1.681717
9,10,actor,1.541909


Category: politics


,rank,word,average_tfidf
0,1,mr,4.245042
1,2,labour,3.985432
2,3,blair,3.378632
3,4,election,3.291674
4,5,party,3.213782
5,6,government,2.770649
6,7,brown,2.647505
7,8,howard,2.255477
8,9,lord,2.187376
9,10,minister,1.996098


Category: sport


,rank,word,average_tfidf
0,1,england,2.033587
1,2,game,1.840940
2,3,cup,1.591267
3,4,wales,1.474310
4,5,rugby,1.436791
5,6,club,1.398004
6,7,ireland,1.394720
7,8,win,1.391316
8,9,match,1.337642
9,10,chelsea,1.328191


Category: tech


,rank,word,average_tfidf
0,1,mobile,3.850275
1,2,technology,2.897042
2,3,phone,2.712343
3,4,users,2.607124
4,5,people,2.483017
5,6,net,2.478178
6,7,games,2.347367
7,8,digital,2.327200
8,9,microsoft,2.243328
9,10,software,2.239569


## High TF with low IDF and the reverse

Here TF is the mean raw term count across all training articles, including zeros. We define low IDF as the bottom 10% of training vocabulary IDFs and high IDF as the top 10%. Within the low-IDF group, we show the 10 highest mean TF terms; within the high-IDF group, the 10 lowest mean TF terms. Ties are sorted alphabetically. These are relative, reproducible selections, not universal thresholds.

In [6]:
print('High mean TF among low-IDF words:')
display(common.reset_index(drop=True))
print('Low mean TF among high-IDF words:')
display(rare.reset_index(drop=True))

High mean TF among low-IDF words:


,word,mean_tf,document_frequency,idf
0,said,3.273746,1435,0.166518
1,mr,1.359292,598,1.041847
2,would,1.174041,866,0.671553
3,year,1.040118,902,0.630823
4,also,0.962832,970,0.558142
5,new,0.920944,760,0.802120
6,people,0.904425,589,1.057012
7,us,0.876696,651,0.956928
8,one,0.864307,816,0.731024
9,could,0.710914,682,0.910408


Low mean TF among high-IDF words:


,word,mean_tf,document_frequency,idf
0,aa,0.00059,1,7.435438
1,aaltra,0.00059,1,7.435438
2,aamir,0.00059,1,7.435438
3,aan,0.00059,1,7.435438
4,aara,0.00059,1,7.435438
5,aarhus,0.00059,1,7.435438
6,abate,0.00059,1,7.435438
7,abatement,0.00059,1,7.435438
8,abbot,0.00059,1,7.435438
9,abbreviated,0.00059,1,7.435438


## Interpretation of measured results

In [7]:
best = metrics.loc[metrics.macro_f1.idxmax()]
print(f"The stronger macro-F1 baseline in this split is {best['representation']}: "
      f"accuracy={best['accuracy']:.4f}, macro F1={best['macro_f1']:.4f}.")
print('Common-word examples:', ', '.join(common.word.head(5)))
print('Rare-word examples:', ', '.join(rare.word.head(5)))
print('Test OOV token rate:', f"{summary['test_oov_token_count']/summary['test_token_count']:.2%}")

The stronger macro-F1 baseline in this split is BoW: accuracy=0.9693, macro F1=0.9686.
Common-word examples: said, mr, would, year, also
Rare-word examples: aa, aaltra, aamir, aan, aara
Test OOV token rate: 3.48%


Common words occur in many articles, making their IDF lower even when their counts are high. Rare words receive large IDF values but may occur only once, so high IDF alone does not guarantee a useful category keyword. The rare-word list can include unusual names and preprocessing fragments. Category averages combine repetition and rarity and tend to highlight topic vocabulary.

A higher TF-IDF score or a TF-IDF representation does not guarantee a better classifier. Naive Bayes naturally models word counts; L2 normalization changes feature scale and the effect of additive smoothing. The experiment reports what happens under the stated fixed settings, without claiming one method is always superior. Both methods discard word order and cannot directly distinguish phrases or contextual meanings.

Results are specific to one reproducible split of this corpus. Duplicate removal prevents exact input leakage, but near duplicates and the small, topic-focused corpus can make performance optimistic for news from another source or period.

## Deliverables and sources

`results/` contains classification metrics, per-class reports, confusion matrices, test predictions, split membership, cleaned articles, word statistics and the top-word tables. `bbc_classification.py` reruns the full experiment.

Sources: the supplied BBC News assignment DOCX, `vector_methods_enhanced_2 (1).pdf`, and `bbc-text (1).csv`.